# When Should Colombo Act? Dengue Outbreak Control Timing with a Calibrated SEIR Model

Dengue is a major public-health burden in Sri Lanka, and Colombo District reports more cases than any other district in most years. This notebook builds a seasonal SEIR (Susceptible–Exposed–Infectious–Recovered) model of dengue in Colombo, solves it with a hand-written fourth-order Runge–Kutta (RK4) method, calibrates it to weekly surveillance data with the Metropolis MCMC algorithm, and uses the fitted model to estimate how many cases a transmission-reducing intervention prevents when it starts earlier rather than later, with credible intervals from the posterior samples.

**Status:** the data, model and solver are complete; calibration, verification checks and the intervention experiments are added in later sections as the project progresses.

*Run with* **Runtime → Run all**. *Everything is downloaded from the public repository; no files or manual steps are needed.*

In [ ]:
# Setup: packages, seeds and data location.
# All of these are pre-installed on Google Colab; the install line makes local runs work too.
%pip install -q --disable-pip-version-check numpy scipy matplotlib pandas

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt

NOTEBOOK_START = time.perf_counter()

SEED = 20261115                # master seed: every random generator below is derived from it
rng = np.random.default_rng(SEED)

REPO_RAW = "https://raw.githubusercontent.com/IT23226814/colombo-dengue-control-timing/main"
FIG_DIR = Path("figures")       # figures are also saved here (relative to the working directory)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Problem and question

Dengue cases in Colombo rise sharply in some years, usually after the monsoon rains. Health officials must decide **when** to start vector-control and clean-up campaigns: acting early costs resources before the outbreak is obvious, while acting late can mean thousands of extra cases.

**Question.** *If an intervention reduces dengue transmission in Colombo District by a given percentage, how many reported cases does it prevent when it starts earlier rather than later?*

The answer is given as cases prevented, with 95% credible intervals, for a range of start weeks and reduction levels, using a model calibrated to the 2026 weekly case series.

**Scope.** One district (Colombo), one season (2026, weeks 1–45), one aggregated intervention that lowers transmission by a fixed fraction from its start week, either for the rest of the year or as a fixed-length campaign. The mosquito population is not modelled explicitly in the core model.

## 2. Model, assumptions and equations

**State variables** (people): $S$ susceptible, $E$ exposed (infected, not yet infectious), $I$ infectious, $R$ recovered or immune, and $C$ the cumulative number of new symptomatic infections. Time $t$ is in **weeks**.

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta(t)\,\frac{S\,I}{N}, &
\frac{dE}{dt} &= \beta(t)\,\frac{S\,I}{N} - \sigma E, &
\frac{dI}{dt} &= \sigma E - \gamma I, \\
\frac{dR}{dt} &= \gamma I, &
\frac{dC}{dt} &= \sigma E, &
N &= S + E + I + R .
\end{aligned}
$$

**Seasonal transmission with an intervention**

$$
\beta(t) = \beta_0\left[1 + a\cos\!\left(\frac{2\pi\,(t-\phi)}{52}\right)\right] u(t),
\qquad
u(t) = \begin{cases} 1 - r, & t_{\text{start}} \le t < t_{\text{end}} \\ 1, & \text{otherwise.} \end{cases}
$$

**Observation model.** New symptomatic infections in week $k$ are $C(k+1) - C(k)$; the expected number of **reported** cases is $\rho\,[C(k+1) - C(k)] + b$, where $\rho$ is the reporting rate and $b$ a constant background of reported cases per week.

**Basic reproduction number.** For a fully susceptible population, $R_0(t) = \beta(t)/\gamma$. If $R_0(t) < 1$ at all times, infections can only decline.

| Symbol | Meaning | Units | Value / source |
|---|---|---|---|
| $N$ | Colombo District population | people | 2,375,415 (Census of Population and Housing 2024) |
| $\beta_0$ | mean transmission rate | 1/week | estimated (MCMC) |
| $a$ | seasonal amplitude | – | estimated (MCMC), $0 \le a < 1$ |
| $\phi$ | week of peak transmission | weeks | estimated (MCMC) |
| $1/\sigma$ | intrinsic incubation period | weeks | 5.9 days = 0.843 weeks [3] |
| $1/\gamma$ | infectious (viraemic) period | weeks | 5 days = 0.714 weeks [4] |
| $\rho$ | reporting rate | – | estimated (MCMC) |
| $b$ | background reported cases | cases/week | estimated (MCMC) |
| $s_0$ | initial susceptible fraction | – | estimated (MCMC) |
| $r$, $t_{\text{start}}$ | intervention strength and start week | –, weeks | scenario inputs |

**Assumptions**
1. *Homogeneous mixing* within the district; no movement between districts.
2. *Host-only model:* transmission through *Aedes* mosquitoes, including the extrinsic incubation in the mosquito (about 8–12 days [4]), is absorbed into $\beta(t)$. This is the main structural simplification; a host–vector extension is a stretch goal.
3. *Seasonality* is a single annual cosine in $\beta$, standing in for rainfall- and temperature-driven mosquito abundance.
4. *Immunity:* only a fraction $s_0$ of the population is susceptible at the start of the year (past exposure to dengue); recovery gives immunity for the rest of the season. Serotypes are not distinguished.
5. *Reporting:* reported cases are a fixed fraction $\rho$ of the season's symptomatic infections plus a constant background $b$ of cases that the single-season dynamics do not produce (endemic transmission in pockets of the district, imported infections, and suspected cases that are not dengue); reporting delays are ignored (the data already use revised weekly counts).
6. *Intervention:* any combination of measures is summarised as a fractional cut $r$ in transmission from week $t_{\text{start}}$.

**Concepts applied:** compartmental epidemic models (SEIR); numerical solution of ODEs with the classical fourth-order Runge–Kutta method and its error order; Bayesian calibration with the Metropolis MCMC algorithm; Monte Carlo propagation of parameter uncertainty into credible intervals.

## 3. Implementation

### 3.1 Data

Weekly suspected dengue cases for Colombo District, taken from the National Dengue Control Unit's *Weekly Dengue Update* reports [1], and the district population from the 2024 census [2]. How the series was extracted and checked is documented in [`data/README.md`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/data/README.md).

*Provenance: data-loading code written with an AI coding agent.*

In [ ]:
cases = pd.read_csv(f"{REPO_RAW}/data/colombo_weekly_cases.csv", parse_dates=["week_start", "week_end"])
population = pd.read_csv(f"{REPO_RAW}/data/colombo_population.csv")
N_COLOMBO = int(population.loc[0, "population"])          # people

obs = cases[cases.year == 2026].sort_values("week").reset_index(drop=True)
WEEKS = len(obs)                                           # number of observed weeks
y_obs = obs["cases"].to_numpy()                            # reported cases per week

# Sanity checks on the input data
assert N_COLOMBO == 2_375_415
assert obs["week"].tolist() == list(range(1, WEEKS + 1)), "2026 weeks must be consecutive from week 1"
assert (y_obs >= 0).all() and not obs["cases"].isna().any()

print(f"Population N = {N_COLOMBO:,}")
print(f"2026 data: weeks 1–{WEEKS} ({obs.week_start.iloc[0].date()} to {obs.week_end.iloc[-1].date()}), "
      f"{y_obs.sum():,} reported cases, peak {y_obs.max():,} in week {obs.week[y_obs.argmax()]}")

fig, ax = plt.subplots()
for year, colour in [(2025, "0.55"), (2026, "C3")]:
    d = cases[cases.year == year].set_index("week")["cases"].reindex(range(1, 53))  # gaps stay empty
    ax.plot(d.index, d.values, "o-", ms=3, color=colour, label=str(year))
ax.set(xlabel="Week of year", ylabel="Reported cases per week", title="Weekly dengue cases, Colombo District", xlim=(0, 53))
ax.legend(frameon=False)
plt.show()

### 3.2 SEIR model and RK4 solver

The model is written from scratch: the right-hand side `rhs`, a classical RK4 step `rk4_step` (four stages, local error $O(h^5)$, global error $O(h^4)$), and `simulate`, which advances the state with a fixed step of $h = 1/7$ week (one day). The cumulative compartment $C$ gives exact weekly incidence as a difference of two RK4 states. Every parameter may be a NumPy array, so many parameter sets (e.g. posterior samples) are simulated in one call.

The same code is kept in [`src/seir.py`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/src/seir.py), where an automated test suite also runs against it; a test checks that this cell and the file stay identical.

*Provenance: written with an AI coding agent; see the project's AI-use record.*

In [ ]:
"""Seasonal SEIR model of dengue transmission, solved with a hand-written RK4.

Units used throughout
---------------------
time            weeks (t = 0 is the start of the first simulated week)
rates           1/week (beta0, sigma, gamma)
compartments    people (S, E, I, R and N)
C               cumulative new symptomatic infections (people), dC/dt = sigma*E

Model
-----
    dS/dt = -beta(t) S I / N
    dE/dt =  beta(t) S I / N - sigma E
    dI/dt =  sigma E - gamma I
    dR/dt =  gamma I
    dC/dt =  sigma E                      (counts every E -> I transition)

    beta(t) = beta0 * [1 + amp * cos(2 pi (t - phase) / period)] * u(t)
    u(t)    = 1 - reduction  while start <= t < end  (intervention), else 1

Weekly new infections in week k (from t = k to k + 1) are C(k+1) - C(k), and
the expected *reported* cases are rho times that.

This is a host-only model: the mosquito stage is not modelled explicitly, so
transmission through mosquitoes is absorbed into beta(t).

Every parameter may be a float or a NumPy array. Arrays are broadcast against
each other, so one call can run many parameter sets (e.g. MCMC samples) at once.
"""

from __future__ import annotations

from dataclasses import dataclass, fields, replace

import numpy as np

DAYS_PER_WEEK = 7.0

# Literature defaults for the natural history (converted to 1/week):
#   mean intrinsic incubation period 5.9 days  [Chan & Johansson 2012, PLoS ONE 7(11):e50972,
#                                               doi:10.1371/journal.pone.0050972]
#   human viraemic (infectious) period ~5 days [WHO dengue fact sheet: "about 4-5 days"]
SIGMA_DEFAULT = DAYS_PER_WEEK / 5.9
GAMMA_DEFAULT = DAYS_PER_WEEK / 5.0


@dataclass(frozen=True)
class Params:
    """Model parameters. Floats or broadcastable NumPy arrays."""

    N: float                      # population (people)
    beta0: float                  # mean transmission rate (1/week)
    amp: float = 0.0              # seasonal amplitude, 0 <= amp < 1 (dimensionless)
    phase: float = 0.0            # week of peak transmission (weeks)
    period: float = 52.0          # length of the seasonal cycle (weeks)
    sigma: float = SIGMA_DEFAULT  # 1 / incubation period (1/week)
    gamma: float = GAMMA_DEFAULT  # 1 / infectious period (1/week)
    rho: float = 1.0              # reporting rate: reported / symptomatic infections
    s0: float = 1.0               # initial susceptible fraction of N
    e0: float = 0.0               # initial exposed (people)
    i0: float = 1.0               # initial infectious (people)
    start: float = np.inf         # intervention start (weeks)
    end: float = np.inf           # intervention end (weeks)
    reduction: float = 0.0        # fractional cut in beta during the intervention

    def with_(self, **changes) -> "Params":
        """Return a copy with some parameters changed."""
        return replace(self, **changes)

    def arrays(self) -> dict[str, np.ndarray]:
        """All parameters as NumPy arrays broadcast to one common shape."""
        names = [f.name for f in fields(self)]
        vals = np.broadcast_arrays(*[np.asarray(getattr(self, n), dtype=float) for n in names])
        return dict(zip(names, vals))


def control(t: float, p: dict[str, np.ndarray]) -> np.ndarray:
    """Intervention multiplier u(t): 1 - reduction while start <= t < end, else 1."""
    active = (t >= p["start"]) & (t < p["end"])
    return np.where(active, 1.0 - p["reduction"], 1.0)


def beta(t: float, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Transmission rate at time t (1/week). u is the intervention multiplier;
    if None it is evaluated at t."""
    seasonal = p["beta0"] * (1.0 + p["amp"] * np.cos(2.0 * np.pi * (t - p["phase"]) / p["period"]))
    return seasonal * (control(t, p) if u is None else u)


def basic_reproduction_number(t: float, params: Params) -> np.ndarray:
    """Instantaneous R0(t) = beta(t) / gamma for a fully susceptible population."""
    p = params.arrays()
    return beta(t, p) / p["gamma"]


def rhs(t: float, y: np.ndarray, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Right-hand side f(t, y); y has shape (5, ...) = (S, E, I, R, C)."""
    S, E, I, R, C = y
    infection = beta(t, p, u) * S * I / p["N"]
    incidence = p["sigma"] * E
    recovery = p["gamma"] * I
    return np.stack([-infection, infection - incidence, incidence - recovery, recovery, incidence])


def rk4_step(f, t: float, y: np.ndarray, h: float, *args) -> np.ndarray:
    """One classical 4th-order Runge-Kutta step of size h for y' = f(t, y, *args)."""
    k1 = f(t, y, *args)
    k2 = f(t + h / 2, y + h / 2 * k1, *args)
    k3 = f(t + h / 2, y + h / 2 * k2, *args)
    k4 = f(t + h, y + h * k3, *args)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def initial_state(p: dict[str, np.ndarray]) -> np.ndarray:
    """(S, E, I, R, C) at t = 0. People not susceptible, exposed or infectious start in R."""
    S = p["s0"] * p["N"] - p["e0"] - p["i0"]
    R = p["N"] - S - p["e0"] - p["i0"]
    if np.any(S < 0):
        raise ValueError("s0 * N must be at least e0 + i0")
    return np.stack([S, p["e0"], p["i0"], R, np.zeros_like(S)])


def simulate(params: Params, weeks: int, steps_per_week: int = 7) -> dict[str, np.ndarray]:
    """Integrate the model for `weeks` weeks with fixed RK4 steps of 1/steps_per_week weeks.

    Returns a dict with
      t                 (weeks*steps_per_week + 1,) time grid in weeks
      y                 (len(t), 5, ...) states S, E, I, R, C on the grid
      weekly_infections (weeks, ...) new symptomatic infections in each week
      weekly_reported   (weeks, ...) expected reported cases = rho * weekly_infections

    The intervention multiplier u is held constant over each step, at its value
    at the step midpoint, so the right-hand side is smooth within every step.
    Start/end times on the step grid (e.g. whole weeks) are therefore handled
    exactly and keep full 4th-order accuracy.
    """
    p = params.arrays()
    h = 1.0 / steps_per_week
    n = weeks * steps_per_week
    y = initial_state(p)
    out = np.empty((n + 1,) + y.shape)
    out[0] = y
    for k in range(n):
        t = k * h
        y = rk4_step(rhs, t, y, h, p, control(t + h / 2, p))
        out[k + 1] = y
    C_weekly = out[::steps_per_week, 4]
    infections = np.diff(C_weekly, axis=0)
    return {
        "t": np.arange(n + 1) * h,
        "y": out,
        "weekly_infections": infections,
        "weekly_reported": p["rho"] * infections,
    }

**Example run (illustrative parameters, not fitted).** The values below only show that the model produces a seasonal outbreak and that an intervention lowers it; the calibrated values come from Section 3.3.

In [ ]:
example = Params(N=N_COLOMBO, beta0=2.8, amp=0.5, phase=22.0, rho=0.1, s0=0.5, e0=1500.0, i0=1500.0)
weeks = np.arange(1, WEEKS + 1)

baseline = simulate(example, WEEKS)
intervention = simulate(example.with_(start=18, reduction=0.25), WEEKS)

fig, ax = plt.subplots()
ax.plot(weeks, baseline["weekly_reported"], label="no intervention")
ax.plot(weeks, intervention["weekly_reported"], label="25% cut in transmission from week 18")
ax.axvline(18, color="0.5", ls=":")
ax.set(xlabel="Week of 2026", ylabel="Expected reported cases per week", title="Illustrative model run (not fitted)")
ax.legend(frameon=False)
plt.show()

prevented = baseline["weekly_reported"].sum() - intervention["weekly_reported"].sum()
print(f"Illustrative only: {prevented:,.0f} reported cases prevented over weeks 1–{WEEKS}")

### 3.3 Calibration with Metropolis MCMC

**Likelihood.** The observed weekly counts $y_k$ are modelled as negative binomial around the model's expected reported cases $\mu_k(\theta) = \rho\,[C(k+1) - C(k)] + b$ (Section 2):

$$
y_k \sim \text{NegBin}(\text{mean } \mu_k,\ \text{dispersion } \kappa), \qquad \operatorname{Var}(y_k) = \mu_k + \mu_k^2/\kappa ,
$$
$$
\log p(y \mid \theta) = \sum_{k}\Big[\log\Gamma(y_k+\kappa) - \log\Gamma(\kappa) - \log y_k! + \kappa\log\tfrac{\kappa}{\kappa+\mu_k} + y_k\log\tfrac{\mu_k}{\kappa+\mu_k}\Big].
$$

A Poisson likelihood ($\kappa \to \infty$) would force the variance to equal the mean, but surveillance counts vary much more than that from week to week (reporting effort, holidays, batch uploads, and the model's own simplifications), so a Poisson fit would be over-confident. The dispersion $\kappa$ is estimated with the other parameters, so the data decide how much extra noise there is.

**Priors** are stated on the natural scale of each parameter. Bounded parameters are sampled on a logit scale and positive ones on a log scale, $z = \operatorname{logit}\!\big(\tfrac{\theta - \ell}{u - \ell}\big)$ or $z = \log\theta$, and the log-Jacobian $\log|d\theta/dz|$ is added to the log-density so the prior on $\theta$ is unchanged. Sampling on $z$ removes the hard boundaries and makes the posterior closer to Gaussian.

**Metropolis algorithm.** For each chain, with current state $z$:

1. propose $z' = z + L\,\varepsilon$ with $\varepsilon \sim \mathcal N(0, I)$ (a symmetric random walk);
2. accept with probability $\alpha = \min\{1, \exp[\log p(z' \mid y) - \log p(z \mid y)]\}$, otherwise stay at $z$.

Because the proposal is symmetric, the proposal densities cancel and this kernel leaves the posterior invariant (detailed balance). **Tuning:** during burn-in the proposal covariance $LL^\top$ is set to $\frac{2.38^2}{d}\hat\Sigma$, where $\hat\Sigma$ is the *within-chain* covariance of the chains' recent samples (deviations from each chain's own mean, so chains that have not yet met do not inflate the steps), and its overall size is nudged towards an acceptance rate of about 0.25. Halfway and three-quarters of the way through burn-in, any chain whose recent mean log-posterior is an outlier far below the others (below $Q_1 - 2\,\mathrm{IQR}$ across chains) is restarted from the current state of a randomly chosen healthy chain, so a chain stuck in a poor local mode does not spoil the run; the number of restarts is reported. After burn-in the proposal is **frozen** and no chain is touched, so the samples that are kept come from a fixed Markov chain; adapting after that point would break the Markov property.

**Convergence diagnostics.** Several chains start from widely spread points. The split-$\hat R$ statistic compares the variance between and within (halves of) chains; values near 1 mean the chains have forgotten their starts and agree. The effective sample size (ESS) accounts for autocorrelation and sets the Monte Carlo error of posterior means, $\text{sd}/\sqrt{\text{ESS}}$.

The sampler, priors, likelihood and diagnostics below are written from scratch (SciPy is used only for $\log\Gamma$). The same code is kept in [`src/mcmc.py`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/src/mcmc.py) with its own automated tests.

*Provenance: written with an AI coding agent.*

In [ ]:
"""Random-walk Metropolis MCMC, priors on transformed scales, convergence
diagnostics, and the posterior for the seasonal SEIR model.

Everything is written from scratch with NumPy; SciPy is used only for the
log-gamma function. Several chains run side by side as rows of one array, so
each iteration evaluates the log-posterior of all chains in one vectorised call
(the SEIR model broadcasts over parameter arrays).

Sampling space
--------------
Each parameter theta is sampled on an unconstrained scale z:
    bounded (lo, hi)  theta = lo + (hi - lo) * logistic(z)
    positive          theta = exp(z)
    real              theta = z
The density on z includes the log-Jacobian |d theta / d z|, so the prior on
theta is exactly the one stated.

Metropolis algorithm (per chain)
--------------------------------
    propose z' = z + L e,  e ~ N(0, I)          (symmetric proposal)
    accept with probability min(1, exp(log p(z') - log p(z)))
During burn-in the proposal covariance L L^T is adapted from the chains'
recent samples (scaled by 2.38^2 / d) and its overall scale is tuned towards
an acceptance rate of about 0.25. Twice during burn-in (at 1/2 and 3/4),
chains whose recent mean log-posterior is an outlier far below the others
(below Q1 - 2 IQR across chains) are restarted from the current state of a
randomly chosen non-outlier chain, so a chain stuck in a poor local mode does
not ruin the run. After burn-in the proposal is frozen and no chain is
touched, so the kept samples come from a fixed Metropolis kernel with the
posterior as its stationary distribution.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from scipy.special import gammaln

# ---------------------------------------------------------------------------
# Priors and transforms
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Prior:
    """Prior for one parameter.

    kind: "uniform" (on lo..hi), "loguniform" (density proportional to
          1/theta on lo..hi, lo > 0: uniform on the log scale), "normal"
          (mu, sd, truncated to lo..hi), "lognormal" (mu, sd of log theta;
          positive), "beta" (a, b on lo..hi).
    The sampling transform follows from the support: bounded -> logit
    (logit of log theta for loguniform), positive -> log, otherwise identity.
    """

    kind: str
    lo: float = -np.inf
    hi: float = np.inf
    mu: float = 0.0
    sd: float = 1.0
    a: float = 1.0
    b: float = 1.0

    @property
    def transform(self) -> str:
        if self.kind == "loguniform":
            return "loglogit"
        if np.isfinite(self.lo) and np.isfinite(self.hi):
            return "logit"
        if self.lo == 0.0 and not np.isfinite(self.hi):
            return "log"
        return "identity"

    def log_density(self, theta: np.ndarray) -> np.ndarray:
        """log p(theta) up to an additive constant; -inf outside the support."""
        theta = np.asarray(theta, dtype=float)
        inside = (theta > self.lo) & (theta < self.hi)
        with np.errstate(divide="ignore", invalid="ignore"):
            if self.kind == "uniform":
                lp = np.zeros_like(theta)
            elif self.kind == "loguniform":
                lp = -np.log(theta)
            elif self.kind == "normal":
                lp = -0.5 * ((theta - self.mu) / self.sd) ** 2
            elif self.kind == "lognormal":
                lp = -np.log(theta) - 0.5 * ((np.log(theta) - self.mu) / self.sd) ** 2
            elif self.kind == "beta":
                x = (theta - self.lo) / (self.hi - self.lo)
                lp = (self.a - 1) * np.log(x) + (self.b - 1) * np.log1p(-x)
            else:
                raise ValueError(f"unknown prior kind {self.kind!r}")
        return np.where(inside, lp, -np.inf)

    def to_theta(self, z: np.ndarray) -> np.ndarray:
        if self.transform == "loglogit":
            a, b = np.log(self.lo), np.log(self.hi)
            return np.exp(a + (b - a) / (1.0 + np.exp(-z)))
        if self.transform == "logit":
            return self.lo + (self.hi - self.lo) / (1.0 + np.exp(-z))
        if self.transform == "log":
            return np.exp(z)
        return z

    def to_z(self, theta: np.ndarray) -> np.ndarray:
        theta = np.asarray(theta, dtype=float)
        if self.transform == "loglogit":
            x = (np.log(theta) - np.log(self.lo)) / (np.log(self.hi) - np.log(self.lo))
            return np.log(x) - np.log1p(-x)
        if self.transform == "logit":
            x = (theta - self.lo) / (self.hi - self.lo)
            return np.log(x) - np.log1p(-x)
        if self.transform == "log":
            return np.log(theta)
        return theta

    def log_jacobian(self, z: np.ndarray) -> np.ndarray:
        """log |d theta / d z|."""
        if self.transform == "loglogit":
            a, b = np.log(self.lo), np.log(self.hi)
            log_theta = a + (b - a) / (1.0 + np.exp(-z))
            return log_theta + np.log(b - a) - np.logaddexp(0.0, z) - np.logaddexp(0.0, -z)
        if self.transform == "logit":
            # d/dz logistic(z) = s (1 - s); computed stably with logaddexp
            return np.log(self.hi - self.lo) - np.logaddexp(0.0, z) - np.logaddexp(0.0, -z)
        if self.transform == "log":
            return z
        return np.zeros_like(z)


@dataclass(frozen=True)
class Model:
    """A named set of priors and a log-likelihood log p(data | theta).

    log_likelihood receives a dict {name: array of shape (n_chains,)} and
    returns an array of shape (n_chains,).
    """

    priors: dict[str, Prior]
    log_likelihood: object = field(repr=False)

    @property
    def names(self) -> list[str]:
        return list(self.priors)

    def to_theta(self, z: np.ndarray) -> dict[str, np.ndarray]:
        return {n: p.to_theta(z[..., j]) for j, (n, p) in enumerate(self.priors.items())}

    def to_z(self, theta: dict[str, np.ndarray]) -> np.ndarray:
        return np.stack([p.to_z(theta[n]) for n, p in self.priors.items()], axis=-1)

    def log_posterior(self, z: np.ndarray) -> np.ndarray:
        """log p(z | data) up to a constant, for z of shape (n_chains, d)."""
        theta = self.to_theta(z)
        lp = np.zeros(z.shape[0])
        for j, (n, p) in enumerate(self.priors.items()):
            lp = lp + p.log_density(theta[n]) + p.log_jacobian(z[:, j])
        ok = np.isfinite(lp)
        if ok.any():
            ll = np.full(z.shape[0], -np.inf)
            sub = {n: v[ok] for n, v in theta.items()}
            ll[ok] = self.log_likelihood(sub)
            lp = np.where(ok, lp + ll, -np.inf)
        return np.where(np.isnan(lp), -np.inf, lp)


# ---------------------------------------------------------------------------
# Likelihood for counts
# ---------------------------------------------------------------------------


def negbin_logpmf(y: np.ndarray, mean: np.ndarray, k: np.ndarray) -> np.ndarray:
    """log P(Y = y) for a negative binomial with the given mean and dispersion k.

    Var(Y) = mean + mean^2 / k, so k -> infinity recovers the Poisson.
    """
    mean = np.maximum(mean, 1e-12)
    return (gammaln(y + k) - gammaln(k) - gammaln(y + 1)
            + k * (np.log(k) - np.log(k + mean)) + y * (np.log(mean) - np.log(k + mean)))


# ---------------------------------------------------------------------------
# Metropolis sampler
# ---------------------------------------------------------------------------


@dataclass
class MCMCResult:
    names: list[str]
    z: np.ndarray            # (n_keep, n_chains, d) kept samples on the sampling scale
    theta: dict              # {name: (n_keep, n_chains)} kept samples on the natural scale
    log_post: np.ndarray     # (n_keep, n_chains)
    acceptance: np.ndarray   # (n_chains,) acceptance rate after burn-in
    proposal_cov: np.ndarray # (d, d) frozen proposal covariance used after burn-in
    burn_in_trace: dict      # {name: (n_burn, n_chains)} for trace plots
    n_reset: int = 0         # outlier chains restarted during burn-in

    def flat(self, name: str) -> np.ndarray:
        """All kept samples of one parameter, chains pooled."""
        return self.theta[name].reshape(-1)


def metropolis(model: Model, z0: np.ndarray, n_burn: int, n_keep: int, rng: np.random.Generator,
               init_scale: float = 0.1, adapt_every: int = 100, target_accept: float = 0.25,
               thin: int = 1, reset_outliers: bool = True) -> MCMCResult:
    """Run len(z0) random-walk Metropolis chains in parallel.

    z0: (n_chains, d) starting points on the sampling scale (use different,
        over-dispersed starts so convergence can be checked).
    n_burn: iterations with proposal adaptation (discarded).
    n_keep: iterations kept after burn-in (before thinning).
    reset_outliers: restart outlier chains during burn-in (see module docstring).
    """
    z = np.array(z0, dtype=float)
    n_chains, d = z.shape
    lp = model.log_posterior(z)
    if not np.all(np.isfinite(lp)):
        raise ValueError("every chain must start where the posterior is finite")

    cov = np.eye(d) * init_scale**2          # proposal covariance before adaptation
    log_scale = 0.0                          # multiplies the covariance by exp(2 * log_scale)
    burn = np.empty((n_burn, n_chains, d))
    burn_lp = np.empty((n_burn, n_chains))
    window_accepts = 0
    n_reset = 0
    checkpoints = {n_burn // 2, (3 * n_burn) // 4} if reset_outliers and n_chains >= 4 else set()

    def step(z, lp, chol):
        proposal = z + rng.standard_normal((n_chains, d)) @ chol.T
        lp_new = model.log_posterior(proposal)
        accept = np.log(rng.random(n_chains)) < lp_new - lp
        z = np.where(accept[:, None], proposal, z)
        lp = np.where(accept, lp_new, lp)
        return z, lp, accept

    chol = np.linalg.cholesky(cov)
    for i in range(n_burn):
        z, lp, acc = step(z, lp, np.exp(log_scale) * chol)
        burn[i] = z
        burn_lp[i] = lp
        window_accepts += acc.sum()
        if (i + 1) in checkpoints:
            recent_lp = burn_lp[i + 1 - n_burn // 4: i + 1].mean(axis=0)   # mean log-posterior per chain
            q1, q3 = np.percentile(recent_lp, [25, 75])
            outliers = np.flatnonzero(recent_lp < q1 - 2.0 * max(q3 - q1, 1.0))
            good = np.setdiff1d(np.arange(n_chains), outliers)
            for c in outliers:
                donor = rng.choice(good)
                z[c], lp[c] = z[donor], lp[donor]
            n_reset += len(outliers)
        if (i + 1) % adapt_every == 0:
            rate = window_accepts / (adapt_every * n_chains)
            log_scale += (rate - target_accept) * 2.0 / np.sqrt((i + 1) / adapt_every)
            window_accepts = 0
            # within-chain covariance of the second half of burn-in so far: deviations from each
            # chain's own mean, pooled over chains, so chains that have not met yet do not
            # inflate the proposal with the distance between them
            recent = burn[(i + 1) // 2: i + 1]                       # (n_recent, n_chains, d)
            if recent.shape[0] * n_chains > 10 * d:
                dev = (recent - recent.mean(axis=0)).reshape(-1, d)
                emp = dev.T @ dev / (dev.shape[0] - n_chains) + 1e-8 * np.eye(d)
                cov = 2.38**2 / d * emp
                chol = np.linalg.cholesky(cov)

    final_chol = np.exp(log_scale) * chol    # frozen from here on
    keep = np.empty((n_keep // thin, n_chains, d))
    keep_lp = np.empty((n_keep // thin, n_chains))
    accepted = np.zeros(n_chains)
    for i in range(n_keep):
        z, lp, acc = step(z, lp, final_chol)
        accepted += acc
        if (i + 1) % thin == 0:
            keep[(i + 1) // thin - 1] = z
            keep_lp[(i + 1) // thin - 1] = lp

    theta = model.to_theta(keep)
    return MCMCResult(
        names=model.names, z=keep, theta=theta, log_post=keep_lp,
        acceptance=accepted / n_keep, proposal_cov=final_chol @ final_chol.T,
        burn_in_trace=model.to_theta(burn), n_reset=n_reset,
    )


# ---------------------------------------------------------------------------
# Convergence diagnostics
# ---------------------------------------------------------------------------


def split_rhat(x: np.ndarray) -> float:
    """Split-R-hat (Gelman et al.) for samples x of shape (n_iter, n_chains).

    Each chain is split in half; values close to 1 (below about 1.01-1.05)
    indicate that the chains agree with each other and with themselves.
    """
    n = x.shape[0] // 2
    halves = np.concatenate([x[:n], x[n:2 * n]], axis=1)       # (n, 2 * n_chains)
    chain_means = halves.mean(axis=0)
    W = halves.var(axis=0, ddof=1).mean()                       # within-chain variance
    B = n * chain_means.var(ddof=1)                             # between-chain variance
    var_plus = (n - 1) / n * W + B / n
    return float(np.sqrt(var_plus / W))


def effective_sample_size(x: np.ndarray) -> float:
    """Effective sample size for x of shape (n_iter, n_chains), using the
    multi-chain autocorrelation and Geyer's initial positive sequence."""
    n, m = x.shape
    centred = x - x.mean(axis=0)
    f = np.fft.rfft(centred, n=2 * n, axis=0)
    acov = np.fft.irfft(f * np.conj(f), axis=0)[:n] / n          # (n, m) autocovariance per chain
    W = x.var(axis=0, ddof=1).mean()
    var_plus = (n - 1) / n * W + x.mean(axis=0).var(ddof=1) if m > 1 else W
    rho = 1.0 - (W - acov.mean(axis=1)) / var_plus
    rho[0] = 1.0
    tau = -1.0
    for t in range(0, n - 1, 2):                                 # sum pairs while positive
        pair = rho[t] + rho[t + 1]
        if pair < 0:
            break
        tau += 2.0 * pair
    return float(n * m / max(tau, 1e-12))


def summarise(result: MCMCResult) -> list[dict]:
    """Posterior mean, 95% credible interval, split-R-hat and ESS for each parameter."""
    rows = []
    for name in result.names:
        x = result.theta[name]
        lo, hi = np.percentile(x, [2.5, 97.5])
        rows.append(dict(parameter=name, mean=x.mean(), sd=x.std(ddof=1), ci_2_5=lo, ci_97_5=hi,
                         rhat=split_rhat(x), ess=effective_sample_size(x)))
    return rows


# ---------------------------------------------------------------------------
# Posterior for the seasonal SEIR model
# ---------------------------------------------------------------------------


def seir_model(y_obs: np.ndarray, priors: dict[str, Prior], fixed: dict, simulate, Params,
               steps_per_week: int = 7) -> Model:
    """Negative-binomial likelihood of weekly reported cases under the SEIR model.

    Expected reported cases in week k are  rho * (new infections in week k) + bg,
    where bg is an optional constant background of reported cases per week.

    priors: priors for any of beta0, amp, phase, rho, s0, i0 (initial
            infectious people), k (negative-binomial dispersion) and bg
            (background reported cases per week; omit it to fix bg = 0).
            Instead of beta0 the prior may be on Re = beta0 * s0 / gamma, the
            mean effective reproduction number at the start; the data fix
            this product much better than beta0 and s0 separately, so
            sampling (Re, s0) avoids a long curved ridge in (beta0, s0).
    fixed:  values for everything not estimated: Params fields (must include
            N) and, optionally, k and bg. Without k (estimated or fixed) the
            likelihood is Poisson.
    Initial exposed people are set to e0 = i0 * gamma / sigma, the ratio at
    which E and I are in balance when incidence is roughly steady.
    simulate, Params: the model's functions, passed in so this module does
    not depend on how the model is imported.
    """
    y_obs = np.asarray(y_obs, dtype=float)
    weeks = len(y_obs)
    observation_fields = {"k", "bg"}
    fixed_model = {n: v for n, v in fixed.items() if n not in observation_fields}
    fixed_obs = {n: v for n, v in fixed.items() if n in observation_fields}

    def log_likelihood(theta: dict) -> np.ndarray:
        kwargs = dict(fixed_model)
        kwargs.update({n: v for n, v in theta.items() if n not in observation_fields | {"Re"}})
        if "Re" in theta:   # mean effective reproduction number Re = beta0 * s0 / gamma
            gamma = kwargs.get("gamma", Params.__dataclass_fields__["gamma"].default)
            kwargs["beta0"] = theta["Re"] * gamma / kwargs["s0"]
        p = Params(**kwargs)
        arr = p.arrays()
        n_chains = len(next(iter(theta.values())))
        i0 = np.broadcast_to(arr["i0"], (n_chains,)).copy()
        ratio = np.broadcast_to(arr["gamma"] / arr["sigma"], (n_chains,))
        # Parameter sets with more initial infected than susceptible people are impossible:
        # give them a harmless value for the run and a log-likelihood of -inf afterwards.
        bad = np.broadcast_to(arr["s0"] * arr["N"], (n_chains,)) < i0 * (1.0 + ratio)
        i0[bad] = 1.0
        p = p.with_(i0=i0, e0=i0 * ratio)
        mean = simulate(p, weeks, steps_per_week)["weekly_reported"]
        mean = np.broadcast_to(mean.reshape(weeks, -1), (weeks, n_chains))  # (weeks, n_chains)
        obs = {**fixed_obs, **{n: v for n, v in theta.items() if n in observation_fields}}
        if "bg" in obs:
            mean = mean + obs["bg"]
        if "k" in obs:
            ll = negbin_logpmf(y_obs[:, None], mean, obs["k"]).sum(axis=0)
        else:                                                          # Poisson likelihood
            m = np.maximum(mean, 1e-12)
            ll = (y_obs[:, None] * np.log(m) - m - gammaln(y_obs[:, None] + 1)).sum(axis=0)
        return np.where(bad, -np.inf, ll)

    return Model(priors=priors, log_likelihood=log_likelihood)

**Sampler check on a target with a known answer.** Before using the sampler on the epidemic model, it is run on a correlated two-dimensional Gaussian whose mean and covariance are known exactly. Six chains start far apart; after burn-in the sample mean must be within 4 Monte Carlo standard errors of the true mean, the covariance must match, split-$\hat R$ must be below 1.02, and the acceptance rate must be in the tuned range.

In [ ]:
# Sampler check: correlated 2-D Gaussian with known mean and covariance
true_mean = np.array([1.0, -2.0])
true_cov = np.array([[1.0, 0.8], [0.8, 2.0]])
true_prec = np.linalg.inv(true_cov)


def gaussian_loglik(theta):
    d = np.stack([theta["x"], theta["y"]], axis=-1) - true_mean
    return -0.5 * np.einsum("ni,ij,nj->n", d, true_prec, d)


gauss = Model(priors={"x": Prior("uniform", lo=-50, hi=50), "y": Prior("uniform", lo=-50, hi=50)},
              log_likelihood=gaussian_loglik)
gauss_rng = np.random.default_rng([SEED, 33])
starts = gauss.to_z({"x": gauss_rng.uniform(-8, 8, 6), "y": gauss_rng.uniform(-8, 8, 6)})
gres = metropolis(gauss, starts, n_burn=3000, n_keep=6000, rng=gauss_rng)

samples = np.stack([gres.flat("x"), gres.flat("y")], axis=1)
for j, name in enumerate(["x", "y"]):
    ess = effective_sample_size(gres.theta[name])
    mc_se = np.sqrt(true_cov[j, j] / ess)
    rhat = split_rhat(gres.theta[name])
    print(f"{name}: mean {samples[:, j].mean():+.3f} (true {true_mean[j]:+.1f}, MC s.e. {mc_se:.3f}), "
          f"R-hat {rhat:.4f}, ESS {ess:,.0f}")
    assert abs(samples[:, j].mean() - true_mean[j]) < 4 * mc_se
    assert rhat < 1.02
print("sample covariance:\n", np.round(np.cov(samples, rowvar=False), 3))
print("acceptance rate per chain:", np.round(gres.acceptance, 3))
assert np.allclose(np.cov(samples, rowvar=False), true_cov, rtol=0.15, atol=0.1)
assert np.all((gres.acceptance > 0.15) & (gres.acceptance < 0.45))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
n_burn_plot = gres.burn_in_trace["x"].shape[0]
for c in range(6):
    trace = np.concatenate([gres.burn_in_trace["x"][:, c], gres.theta["x"][:, c]])
    axes[0].plot(trace, lw=0.4)
axes[0].axvline(n_burn_plot, color="k", ls="--", lw=0.8)
axes[0].set(xlabel="Iteration", ylabel="x", title="Trace of x for 6 chains (dashed: end of burn-in)")
axes[1].plot(samples[::10, 0], samples[::10, 1], ".", ms=1.5, alpha=0.4)
axes[1].set(xlabel="x", ylabel="y", title="Kept samples (every 10th)")
plt.tight_layout()
plt.show()
print("PASS: the sampler recovers the known mean and covariance, and the chains agree")

**Posterior for the SEIR model.** The parameters estimated from the data and their priors:

| Parameter | Meaning | Prior | Reasoning |
|---|---|---|---|
| $R_e = \beta_0 s_0/\gamma$ | mean effective reproduction number at the start of the year | Uniform(0.2, 5) | from no epidemic potential to very strong transmission; $\beta_0 = R_e\gamma/s_0$ |
| $a$ | seasonal amplitude | Uniform(0, 0.95) | from no seasonality to near-complete seasonal swing; must be below 1 to keep $\beta>0$ |
| $\phi$ | week of peak transmission | Uniform(10, 40) | transmission peaks before the case peak; the data peak in week 27 |
| $\rho$ | reporting rate | LogNormal, median 0.132, log-sd 0.5, truncated to $(0,1)$ | about 13% of symptomatic dengue episodes are reported in Southeast Asia, with country estimates from about 5% to 26% [5]; the prior's central 95% range is 0.05–0.35 |
| $s_0$ | initial susceptible fraction | Uniform(0.01, 1) | unknown share of Colombo residents who are both without immunity and exposed to transmission |
| $I_0$ | infectious people at week 0 | LogNormal, median 1,000, log-sd 1.5 | wide; sets the size of the early-year epidemic; $E_0 = I_0\gamma/\sigma$ |
| $\kappa$ | negative-binomial dispersion | LogNormal, median 20, log-sd 1.5 | from strong to mild overdispersion |
| $b$ | background reported cases per week | Uniform(0, 1000) | steady endemic/imported cases not produced by the season's outbreak dynamics |

**Why $R_e$ and an informative prior on $\rho$.** One season of case counts mostly determines the *shape* of the outbreak, which depends on $\beta_0$ and $s_0$ almost only through their product, and on how far the susceptible pool is depleted, which depends on $\rho s_0$. Sampling $\beta_0$ and $s_0$ directly produces a long, curved ridge ($\beta_0 \propto 1/s_0$) that a random-walk sampler crosses very slowly; in early runs the chains did not converge ($\hat R$ above 2). Sampling $R_e$ and $s_0$ straightens that ridge. Likewise, with a flat prior on $\rho$ the data allowed two explanations of the outbreak's end — few infections reported and the outbreak stopped by depletion, or many reported and stopped by the season — and the chains split between them. The literature-based prior on $\rho$ resolves this, and the effect of the choice is examined in the sensitivity analysis. Section 4.4 shows that the resulting sampler recovers known parameters.

The incubation and infectious periods stay fixed at their literature values (Section 2). The posterior is built by `seir_model(...)` from the code above; it is run on synthetic data in Section 4.4 and on the 2026 Colombo data in Section 3.4.

In [ ]:
# Priors for the SEIR calibration (natural scale)
SEIR_PRIORS = {
    "Re":    Prior("uniform", lo=0.2, hi=5.0),                   # beta0 * s0 / gamma
    "amp":   Prior("uniform", lo=0.0, hi=0.95),
    "phase": Prior("uniform", lo=10.0, hi=40.0),                 # weeks
    "rho":   Prior("lognormal", lo=0.0, hi=1.0, mu=np.log(0.132), sd=0.5),
    "s0":    Prior("uniform", lo=0.01, hi=1.0),
    "i0":    Prior("lognormal", lo=0.0, mu=np.log(1000.0), sd=1.5),  # people
    "k":     Prior("lognormal", lo=0.0, mu=np.log(20.0), sd=1.5),
    "bg":    Prior("uniform", lo=0.0, hi=1000.0),                # reported cases per week
}
FIXED = dict(N=N_COLOMBO)                 # sigma, gamma: literature defaults
MCMC_STEPS_PER_WEEK = 2                   # RK4 step of half a week: error < 1 person (Section 4.3)


def colombo_posterior(y):
    # Log-posterior of the SEIR parameters given weekly reported cases y
    return seir_model(y, SEIR_PRIORS, FIXED, simulate, Params, MCMC_STEPS_PER_WEEK)


def beta0_from(theta):
    # Transmission rate implied by Re and s0 (1/week)
    return theta["Re"] * GAMMA_DEFAULT / theta["s0"]


def dispersed_starts(model, n_chains, rng):
    # Over-dispersed starting points drawn from broad ranges, keeping only finite posteriors
    starts = []
    while len(starts) < n_chains:
        theta = {"Re": rng.uniform(0.6, 2.5), "amp": rng.uniform(0.05, 0.8), "phase": rng.uniform(12, 35),
                 "rho": np.exp(rng.normal(np.log(0.13), 0.5)), "s0": rng.uniform(0.1, 0.95),
                 "i0": np.exp(rng.normal(np.log(500), 1.0)), "k": np.exp(rng.normal(np.log(20), 0.5)),
                 "bg": rng.uniform(50, 500)}
        z = model.to_z({n: np.array([v]) for n, v in theta.items()})
        if np.isfinite(model.log_posterior(z))[0]:
            starts.append(z[0])
    return np.array(starts)


# Quick check: the log-posterior is finite at example points and works for many chains at once
post = colombo_posterior(y_obs)
z_try = post.to_z({"Re": np.array([1.0, 1.2]), "amp": np.array([0.5, 0.3]), "phase": np.array([22.0, 25.0]),
                   "rho": np.array([0.1, 0.2]), "s0": np.array([0.5, 0.4]), "i0": np.array([1500.0, 800.0]),
                   "k": np.array([20.0, 10.0]), "bg": np.array([1.0, 250.0])})
print("log-posterior at two example points:", np.round(post.log_posterior(z_try), 1))
assert np.all(np.isfinite(post.log_posterior(z_try)))

### 3.4 Fit to the 2026 Colombo data

The sampler is run on the 45 weeks of 2026 data with the priors of Section 3.3: 16 chains from over-dispersed starts, 15,000 burn-in iterations with adaptation, then 30,000 iterations with a frozen proposal, keeping every 15th (32,000 posterior samples in total). Convergence is checked with split-$\hat R$ (asserted below 1.1) and the effective sample size, and by trace plots.

Besides the sampled parameters, the table reports quantities derived from each posterior sample:
- $\beta_0 = R_e\gamma/s_0$ and the basic reproduction number for a fully susceptible population, $R_0 = \beta_0/\gamma$;
- the peak effective reproduction number in the season, $R_e(1+a)$;
- $\rho s_0$ and the at-risk population $s_0 N$, which together set how many reported cases it takes to exhaust the susceptible pool.

*Provenance: written with an AI coding agent.*

In [ ]:
# 3.4 Calibration to the 2026 weekly data
fit_rng = np.random.default_rng([SEED, 34])
colombo_post = colombo_posterior(y_obs)
t0 = time.perf_counter()
fit = metropolis(colombo_post, dispersed_starts(colombo_post, 16, fit_rng),
                 n_burn=15_000, n_keep=30_000, rng=fit_rng, thin=15)
print(f"MCMC: 16 chains x 45,000 iterations in {time.perf_counter() - t0:.0f} s; "
      f"acceptance {fit.acceptance.min():.2f}-{fit.acceptance.max():.2f}; "
      f"outlier chains restarted during burn-in: {fit.n_reset}")

draws = {n: fit.flat(n) for n in fit.names}                 # pooled posterior samples
draws["beta0"] = beta0_from(draws)                          # 1/week
derived = {
    "beta0 (1/week)": draws["beta0"],
    "R0 = beta0/gamma": draws["beta0"] / GAMMA_DEFAULT,
    "peak Re = Re(1+a)": draws["Re"] * (1 + draws["amp"]),
    "rho * s0": draws["rho"] * draws["s0"],
    "at-risk people s0*N": draws["s0"] * N_COLOMBO,
}

fit_table = pd.DataFrame(summarise(fit)).set_index("parameter")
for name, x in derived.items():
    lo, hi = np.percentile(x, [2.5, 97.5])
    fit_table.loc[name, ["mean", "sd", "ci_2_5", "ci_97_5"]] = [x.mean(), x.std(ddof=1), lo, hi]
with pd.option_context("display.float_format", lambda v: f"{v:,.4g}"):
    print(fit_table.to_string())

assert (fit_table.rhat.dropna() < 1.1).all(), "chains have not converged (split R-hat >= 1.1)"
print(f"\nPASS: all split R-hat <= {fit_table.rhat.max():.3f}; smallest ESS = {fit_table.ess.min():,.0f}")

# Trace plots (burn-in grey, kept samples coloured) for four key parameters
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
n_b = fit.burn_in_trace["Re"].shape[0]
kept_x = n_b + np.arange(1, fit.theta["Re"].shape[0] + 1) * 15
for ax, name in zip(axes.flat, ["Re", "amp", "phase", "s0"]):
    ax.plot(np.arange(n_b), fit.burn_in_trace[name], color="0.75", lw=0.3)
    ax.plot(kept_x, fit.theta[name], lw=0.4)
    ax.axvline(n_b, color="k", ls="--", lw=0.8)
    ax.set(title=name, ylabel=name)
for ax in axes[1]:
    ax.set_xlabel("Iteration (dashed: end of burn-in)")
fig.suptitle("MCMC traces for the 2026 fit: 16 chains")
plt.tight_layout()
fig.savefig(FIG_DIR / "mcmc_traces.png", dpi=150, bbox_inches="tight")
plt.show()

**Posterior predictive check.** For 2,000 posterior samples the model is simulated (all in one vectorised call, with the one-day step), and a weekly count is drawn from the negative binomial around each simulated mean. The median and 95% band of these predicted counts are compared with the data. If the model and noise model are adequate, about 95% of the observed weeks should fall inside the band, without long runs of weeks on one side.

In [ ]:
# Posterior predictive check
pp_rng = np.random.default_rng([SEED, 35])
pick = pp_rng.choice(draws["Re"].size, size=2000, replace=False)
sample = {n: v[pick] for n, v in draws.items()}


def params_from(sample, **changes):
    # Params for a batch of posterior samples (E0 = I0 * gamma / sigma, as in the likelihood)
    p = Params(N=N_COLOMBO, beta0=sample["beta0"], amp=sample["amp"], phase=sample["phase"],
               rho=sample["rho"], s0=sample["s0"], i0=sample["i0"],
               e0=sample["i0"] * GAMMA_DEFAULT / SIGMA_DEFAULT)
    return p.with_(**changes) if changes else p


pp_mean = simulate(params_from(sample), WEEKS)["weekly_reported"] + sample["bg"]     # (weeks, 2000)
pp_counts = pp_rng.negative_binomial(sample["k"], sample["k"] / (sample["k"] + pp_mean))
band = np.percentile(pp_counts, [2.5, 50, 97.5], axis=1)
mean_band = np.percentile(pp_mean, [2.5, 97.5], axis=1)
inside = (y_obs >= band[0]) & (y_obs <= band[2])
print(f"weeks inside the 95% predictive band: {inside.sum()} of {WEEKS} ({inside.mean():.0%})")
print("weeks outside:", ", ".join(f"wk {w} ({'above' if y > hi else 'below'})"
                                   for w, y, hi, ok in zip(weeks, y_obs, band[2], inside) if not ok) or "none")
peak_week_pred = np.argmax(pp_mean, axis=0) + 1
print(f"predicted peak week: median {np.median(peak_week_pred):.0f} "
      f"(95% interval {np.percentile(peak_week_pred, 2.5):.0f}-{np.percentile(peak_week_pred, 97.5):.0f}); "
      f"observed peak week {weeks[y_obs.argmax()]}")
print(f"predicted total for weeks 1-{WEEKS}: median {np.median(pp_counts.sum(axis=0)):,.0f} "
      f"(95% {np.percentile(pp_counts.sum(axis=0), 2.5):,.0f}-{np.percentile(pp_counts.sum(axis=0), 97.5):,.0f}); "
      f"observed {y_obs.sum():,}")

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.fill_between(weeks, band[0], band[2], color="C0", alpha=0.2, label="95% predictive band (with noise)")
ax.fill_between(weeks, mean_band[0], mean_band[1], color="C0", alpha=0.4, label="95% band of the model mean")
ax.plot(weeks, band[1], color="C0", lw=1.5, label="predictive median")
ax.plot(weeks, y_obs, "o", color="C3", ms=4, label="reported cases 2026")
ax.plot(weeks[~inside], y_obs[~inside], "o", mfc="none", mec="k", ms=9, label="outside the 95% band")
ax.axhline(np.median(sample["bg"]), color="0.4", ls=":", lw=1, label="background b (median)")
ax.set(xlabel="Week of 2026", ylabel="Reported cases per week", title="Posterior predictive check: Colombo District, 2026")
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.savefig(FIG_DIR / "calibration_fit.png", dpi=150, bbox_inches="tight")
plt.show()

**How well does the model fit?** With the seed used here, 43 of the 45 weeks (96%) lie inside the 95% posterior predictive band, as expected for an adequate model. The predicted peak is in week 28 (95% interval 27–28) against an observed week 27, and the predicted total for weeks 1–45 is 25,648 (23,857–27,450) against 25,511 reported. The two weeks outside the band (weeks 36 and 37) lie just below it: after the outbreak the counts dipped below the constant background for a few weeks. During the steep rise (weeks 24–25) the reported counts are above the predictive median, so the model's rise is slightly slower than the real one. These are the main systematic misfits; the early-year decline, the outbreak peak and the decline after it are reproduced well.

**What do the parameters say?**
- *Transmission.* The mean effective reproduction number at the start of the year is $R_e \approx 1.96$ (95% CrI 1.78–2.14), rising to about 3.1 (2.8–3.4) at the seasonal maximum. Seasonal forcing is strong ($a \approx 0.58$) and transmission peaks around week 33 (31.5–34.1).
- *The outbreak ends by depletion, not by the season.* The case peak (week 27) comes *before* the peak in transmission, so in the fitted model the outbreak turns over because the pool of susceptible people who take part in transmission runs out, while the seasonal transmission rate is still rising. This matters for interventions: measures that slow transmission can also delay depletion, so their benefit depends on timing.
- *A small at-risk population.* The fitted susceptible fraction is $s_0 \approx 0.07$ (0.025–0.16), i.e. roughly 170,000 (58,000–380,000) people out of 2.4 million. This should not be read as "93% of Colombo is immune". A model with homogeneous mixing represents a district where transmission is concentrated in some neighbourhoods (the weekly reports flag a few dozen high-risk MOH areas) as a small, well-mixed population. For the same reason the implied basic reproduction number for a fully susceptible district, $\beta_0/\gamma \approx 35$ (12–80), is an artefact of the homogeneous-mixing assumption and is not a biological $R_0$; $R_e$ is the meaningful quantity.
- *Reporting.* The data alone do not separate $\rho$ from $s_0$: the posterior of $\rho$ (0.04–0.28) is close to its literature prior, but their product is tightly determined, $\rho s_0 \approx 0.0070$ (0.0063–0.0078). Equivalently, about $\rho s_0 N \approx 16{,}600$ reported cases would exhaust the at-risk pool.
- *Background.* About $b \approx 247$ (224–272) reported cases per week, roughly 11,000 of the 25,511 reported cases in weeks 1–45, are not explained by the season's outbreak dynamics.
- *Noise.* The dispersion $\kappa \approx 63$ (36–101) means the week-to-week scatter is moderately larger than Poisson.

The posterior samples `draws` from this fit are used for the intervention experiments in Section 5.

## 4. Verification

Each check below ends in an `assert`, so *Run all* stops with an error if the simulator is wrong. The checks test the solver against facts that must hold whatever the parameter values are, so they do not depend on the calibration.

### 4.1 Conservation of population

The four compartment equations sum to zero, $\frac{d}{dt}(S+E+I+R) = 0$, so the total population must stay at $N$ for all time. RK4 is a linear combination of right-hand-side evaluations, so it preserves this linear invariant exactly apart from floating-point rounding. A second consistency check follows from $C' = \sigma E$ and $S' + E' = -\sigma E$: the cumulative count of new infections must equal the number of people who have left $S$ and $E$, $C(t) = [S(0)-S(t)] - [E(t)-E(0)]$.

The check runs 200 random parameter sets at once (wide ranges of $\beta_0$, $a$, $\phi$, $s_0$ and intervention settings) over two years with the daily step used throughout, and requires the relative error to stay below $10^{-12}$ at every step. All states must also stay non-negative.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.1 Conservation: S + E + I + R = N at every RK4 step, for many random parameter sets
check_rng = np.random.default_rng([SEED, 41])        # independent stream for this check
n_sets = 200
random_params = Params(
    N=N_COLOMBO,
    beta0=check_rng.uniform(0.5, 4.0, n_sets),        # 1/week
    amp=check_rng.uniform(0.0, 0.9, n_sets),
    phase=check_rng.uniform(0, 52, n_sets),           # weeks
    s0=check_rng.uniform(0.1, 1.0, n_sets),
    e0=check_rng.uniform(0, 2000, n_sets),            # people
    i0=check_rng.uniform(1, 2000, n_sets),            # people
    start=check_rng.integers(5, 40, n_sets).astype(float),
    reduction=check_rng.uniform(0.0, 0.8, n_sets),
)
run = simulate(random_params, weeks=104)
S, E, I, R, C = (run["y"][:, k] for k in range(5))  # each has shape (steps + 1, n_sets)

total = S + E + I + R
max_rel_error = np.max(np.abs(total - N_COLOMBO) / N_COLOMBO)
C_expected = (S[0] - S) - (E - E[0])
max_C_error = np.max(np.abs(C - C_expected) / N_COLOMBO)
min_state = run["y"][:, :4].min()

print(f"{n_sets} parameter sets x {run['t'].size} time points")
print(f"max |S+E+I+R - N| / N          = {max_rel_error:.2e}")
print(f"max |C - (people left S and E)| / N = {max_C_error:.2e}")
print(f"smallest value of any compartment = {min_state:.3g} people")

assert max_rel_error < 1e-12, "population is not conserved"
assert max_C_error < 1e-12, "cumulative infections do not match the flows out of S and E"
assert min_state > -1e-6, "a compartment became negative"
print("PASS: population conserved, cumulative infections consistent, all states non-negative")

### 4.2 Limiting case: no outbreak when $R_0 < 1$

If every infectious person causes fewer than one new infection on average, an outbreak cannot grow. In this model an infectious person stays infectious for an exponential time with rate $\gamma$ and infects at rate at most $\beta_{\max} = \beta_0(1+a)$ (when everyone is susceptible), so the expected number of new infections per infectious person is at most

$$R_{0,\max} = \frac{\beta_0\,(1+a)}{\gamma}.$$

When $R_{0,\max} < 1$ this gives two testable consequences, starting from $E_0$ exposed and $I_0$ infectious people in a fully susceptible population ($s_0 = 1$):

1. **No growth.** Weekly new infections decline after the first week (constant $\beta$).
2. **A bound on the final size.** Counting every chain of transmission, the number of new infections can be at most $R_{0,\max}(E_0 + I_0)/(1 - R_{0,\max})$, so the cumulative count satisfies
$$C(t) \le E_0 + \frac{R_{0,\max}\,(E_0 + I_0)}{1 - R_{0,\max}} \quad \text{for all } t,$$
with or without seasonality.

As a control, the same run with $R_0 = 1.5$ must produce a large outbreak, showing the check can tell the two regimes apart.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.2 Limiting case R0 < 1: infections decline and stay below the branching-process bound
gamma = GAMMA_DEFAULT                     # 1/week
R0_max = 0.9
seed_people = dict(e0=100.0, i0=100.0)    # people
bound = seed_people["e0"] + R0_max * (seed_people["e0"] + seed_people["i0"]) / (1 - R0_max)

print(f"R0,max = {R0_max}; bound on cumulative infections = {bound:,.0f} people")
for amp in [0.0, 0.3, 0.6]:
    p_sub = Params(N=N_COLOMBO, beta0=R0_max * gamma / (1 + amp), amp=amp, phase=22.0, s0=1.0, **seed_people)
    assert np.isclose(basic_reproduction_number(np.linspace(0, 52, 521), p_sub).max(), R0_max, rtol=1e-3)
    out = simulate(p_sub, weeks=52)
    weekly = out["weekly_infections"]
    C_final = out["y"][-1, 4]
    print(f"  amplitude {amp:.1f}: cumulative infections after 52 weeks = {C_final:7,.1f}, "
          f"largest week = {weekly.max():6.1f}")
    assert np.all(out["y"][:, 4] <= bound + 1e-9), "cumulative infections exceed the R0 < 1 bound"
    if amp == 0.0:
        assert np.all(np.diff(weekly[1:]) < 0), "weekly infections grew although R0 < 1"

# Control: the same seeding with R0 = 1.5 does produce an outbreak
p_super = Params(N=N_COLOMBO, beta0=1.5 * gamma, s0=1.0, **seed_people)
weekly_super = simulate(p_super, weeks=52)["weekly_infections"]
print(f"  control R0 = 1.5: {weekly_super.sum():,.0f} infections in 52 weeks (peak week {weekly_super.max():,.0f})")
assert weekly_super.sum() > 100 * bound, "control run should give an outbreak far above the R0 < 1 bound"

fig, ax = plt.subplots()
ax.semilogy(np.arange(1, 53), simulate(Params(N=N_COLOMBO, beta0=R0_max * gamma, s0=1.0, **seed_people), 52)["weekly_infections"],
            label=f"$R_0$ = {R0_max} (no outbreak)")
ax.semilogy(np.arange(1, 53), weekly_super, label="$R_0$ = 1.5 (outbreak)")
ax.set(xlabel="Week", ylabel="New infections per week (log scale)", title="Limiting case: $R_0$ below and above 1")
ax.legend(frameon=False)
plt.show()
print("PASS: with R0 < 1 infections decline and stay below the bound; with R0 > 1 an outbreak occurs")

### 4.3 Convergence order of the RK4 solver

The classical RK4 method has local truncation error $O(h^5)$ per step and, over a fixed time interval, global error $O(h^4)$: halving the step $h$ should cut the error by about $2^4 = 16$. For comparison, the forward Euler method is first order ($O(h)$) and the midpoint RK2 method second order ($O(h^2)$).

**Set-up.** The illustrative scenario from Section 3.2, *including* an intervention that switches on at week 18, is integrated for 45 weeks with steps $h = 1, \tfrac12, \tfrac14, \dots$ weeks. The error is the largest difference over all weeks in the cumulative number of infections $C$ (people) from a **reference solution** computed independently with SciPy's 8th-order Dormand–Prince method (`DOP853`) at relative tolerance $10^{-13}$, restarted at the switch-on time so that it never steps across the jump in $\beta$. SciPy is used only for this check; the model itself is solved with the hand-written RK4.

**Pass criterion.** The slope of $\log(\text{error})$ against $\log h$ for RK4, fitted over $h = \tfrac18$ to $\tfrac1{64}$ week (where the error is well above the reference's own accuracy), must lie between 3.8 and 4.2. Euler and RK2 must show slopes near 1 and 2.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.3 Error scaling of Euler, RK2 and RK4 against an independent high-accuracy reference
from scipy.integrate import solve_ivp

conv_params = example.with_(start=18, reduction=0.25)   # illustrative scenario with an intervention
P_conv = conv_params.arrays()
CONV_WEEKS = 45


def euler_step(f, t, y, h, *args):
    return y + h * f(t, y, *args)


def rk2_step(f, t, y, h, *args):                          # explicit midpoint method
    k1 = f(t, y, *args)
    return y + h * f(t + h / 2, y + h / 2 * k1, *args)


def integrate(stepper, steps_per_week):
    # Same scheme as simulate(), with any one-step method; returns the state at the end of each week
    h = 1.0 / steps_per_week
    y = initial_state(P_conv)
    weekly = [y]
    for k in range(CONV_WEEKS * steps_per_week):
        t = k * h
        y = stepper(rhs, t, y, h, P_conv, control(t + h / 2, P_conv))
        if (k + 1) % steps_per_week == 0:
            weekly.append(y)
    return np.array(weekly)


# The RK4 path through integrate() must be exactly the model's own solver
assert np.array_equal(integrate(rk4_step, 7), simulate(conv_params, CONV_WEEKS)["y"][::7])

# Reference: DOP853, restarted at the intervention start so it never steps across the jump
y_start = initial_state(P_conv)
reference = [y_start]
for t_a, t_b in [(0, 18), (18, CONV_WEEKS)]:
    u = control((t_a + t_b) / 2, P_conv)
    sol = solve_ivp(lambda t, y: rhs(t, y, P_conv, u), (t_a, t_b), y_start, method="DOP853",
                    rtol=1e-13, atol=1e-9, t_eval=np.arange(t_a + 1, t_b + 1))
    reference.extend(sol.y.T)
    y_start = sol.y[:, -1]
reference = np.array(reference)


def max_error(states):                                  # people, cumulative infections C
    return np.max(np.abs(states[:, 4] - reference[:, 4]))


methods = {"Euler": (euler_step, [8, 16, 32, 64, 128, 256], 1),
           "RK2":   (rk2_step,   [2, 4, 8, 16, 32, 64], 2),
           "RK4":   (rk4_step,   [1, 2, 4, 8, 16, 32, 64], 4)}
results = {}
for name, (stepper, spws, order) in methods.items():
    h = 1.0 / np.array(spws)
    err = np.array([max_error(integrate(stepper, s)) for s in spws])
    in_range = h <= 1 / 8 if name == "RK4" else np.ones_like(h, dtype=bool)
    slope = np.polyfit(np.log(h[in_range]), np.log(err[in_range]), 1)[0]
    results[name] = (h, err, slope, order)
    ratios = err[:-1] / err[1:]
    print(f"{name:5s} fitted order {slope:4.2f} (theory {order}); error ratio per halving: "
          + ", ".join(f"{r:.1f}" for r in ratios))

slope_rk4 = results["RK4"][2]
assert 3.8 <= slope_rk4 <= 4.2, f"RK4 order {slope_rk4:.2f} is outside [3.8, 4.2]"
assert 0.9 <= results["Euler"][2] <= 1.1 and 1.9 <= results["RK2"][2] <= 2.1

default_err = max_error(integrate(rk4_step, 7))
print(f"\nRK4 error at the default step h = 1/7 week (1 day): {default_err:.1e} people "
      f"of {reference[-1, 4]:,.0f} cumulative infections")

fig, ax = plt.subplots(figsize=(7, 5))
for (name, (h, err, slope, order)), marker in zip(results.items(), "osD"):
    ax.loglog(h, err, marker + "-", label=f"{name}: fitted order {slope:.2f}")
    ax.loglog(h, err[-1] * (h / h[-1]) ** order, ":", color="0.5")
ax.axvline(1 / 7, color="C3", lw=0.8, ls="--", label="default step (1 day)")
ax.set(xlabel="Step size h (weeks)", ylabel="Max error in cumulative infections (people)",
       title="Convergence order: dotted guides have slopes 1, 2 and 4")
ax.legend(frameon=False)
fig.savefig(FIG_DIR / "rk4_convergence.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"PASS: RK4 converges at order {slope_rk4:.2f} (expected 4); Euler and RK2 at orders 1 and 2")

The error falls by a factor close to 16 each time the step is halved, as fourth-order theory predicts. At large steps the observed order is above 4 because the leading $h^4$ term does not yet dominate (pre-asymptotic range), which is why the slope is fitted on the smaller steps. The intervention switch does not reduce the order, because the step grid is aligned with the switch-on time and $u(t)$ is held constant within each step. At the one-day step used throughout this notebook the error is a tiny fraction of one person, so the solver error is negligible next to the uncertainty in the data and parameters.

### 4.4 Recovery of known parameters by MCMC

A calibration method is only trustworthy if it finds the right answer when the right answer is known. Synthetic weekly case counts are generated from the model with **known parameter values** (chosen to give an outbreak of similar size and timing to 2026), with negative-binomial noise from a fixed seed. The sampler is then run on these synthetic data exactly as it will be on the real data: same priors, 16 chains from over-dispersed starting points, burn-in with adaptation, then a frozen proposal.

**Pass criteria:** every true value lies inside its 95% credible interval, and every split-$\hat R$ is below 1.1, so the chains agree.

*Provenance: written with an AI coding agent.*

In [ ]:
# 4.4 Recovery of known parameters from synthetic data
truth = dict(Re=3.0 * 0.5 / GAMMA_DEFAULT, amp=0.4, phase=20.0, rho=0.1, s0=0.5, i0=300.0, k=50.0, bg=250.0)
p_true = Params(N=N_COLOMBO, beta0=beta0_from(truth), amp=truth["amp"], phase=truth["phase"], rho=truth["rho"],
                s0=truth["s0"], i0=truth["i0"], e0=truth["i0"] * GAMMA_DEFAULT / SIGMA_DEFAULT)
mean_true = simulate(p_true, WEEKS)["weekly_reported"] + truth["bg"]
synth_rng = np.random.default_rng([SEED, 44])
y_synth = synth_rng.negative_binomial(truth["k"], truth["k"] / (truth["k"] + mean_true))
print(f"synthetic data: {y_synth.sum():,} cases, peak {y_synth.max():,} in week {y_synth.argmax() + 1}")

synth_post = colombo_posterior(y_synth)
t0 = time.perf_counter()
synth_fit = metropolis(synth_post, dispersed_starts(synth_post, 16, synth_rng),
                       n_burn=15_000, n_keep=30_000, rng=synth_rng, thin=15)
print(f"MCMC: 16 chains x 45,000 iterations in {time.perf_counter() - t0:.0f} s; "
      f"acceptance {synth_fit.acceptance.min():.2f}-{synth_fit.acceptance.max():.2f}; "
      f"outlier chains restarted during burn-in: {synth_fit.n_reset}")

table = pd.DataFrame(summarise(synth_fit)).set_index("parameter")
table.insert(0, "true", [truth[n] for n in table.index])
table["inside 95% CI"] = (table.ci_2_5 <= table.true) & (table.true <= table.ci_97_5)
print(table.round(3).to_string())

assert table["inside 95% CI"].all(), "a true value is outside its 95% credible interval"
assert (table.rhat < 1.1).all(), "chains have not converged (split R-hat >= 1.1)"

fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
for ax, name in zip(axes.flat, synth_fit.names):
    ax.hist(synth_fit.flat(name), bins=40, density=True, color="C0", alpha=0.6)
    ax.axvline(truth[name], color="C3", lw=2, label="true value")
    lo, hi = table.loc[name, ["ci_2_5", "ci_97_5"]]
    ax.axvspan(lo, hi, color="0.85", zorder=0)
    ax.set_title(name)
    ax.set_yticks([])
axes.flat[0].legend(frameon=False, fontsize=8)
fig.suptitle("Posterior from synthetic data (grey: 95% credible interval, red: true value)")
plt.tight_layout()
fig.savefig(FIG_DIR / "mcmc_recovery.png", dpi=150, bbox_inches="tight")
plt.show()
print("PASS: every true parameter lies inside its 95% credible interval and the chains have converged")

All eight true values are recovered within their 95% credible intervals. The posterior is narrow for the quantities that shape the epidemic curve ($R_e$, $a$, $\phi$, $b$, $\kappa$) and wide for $s_0$, which stays close to its prior: one season of counts says little about the size of the susceptible pool. This is an honest limitation of fitting to a single season, and the intervention results will be checked for sensitivity to $s_0$.

## 5. Experiments and results

### 5.1 Intervention timing scenarios

**Design.** The question is answered as a counterfactual for 2026: *had Colombo started an intervention in week $w$ that cuts transmission by $r$, how many reported cases would it have prevented?*

- **Interventions.** Two kinds, both entering the model through $u(t) = 1 - r$ (Section 2):
  - *sustained*: from the start of week $w$ until the end of the year;
  - *8-week campaign*: from the start of week $w$ for 8 weeks, a typical length of an intensive vector-control and clean-up drive.
- **Grid.** Start weeks $w = 1, \dots, 40$, and reductions $r$ = 10%, 25% and 40% (plus 5%–50% in steps of 5% for the heatmap).
- **Outcome.** Reported cases prevented in weeks 1–52 of 2026: $\rho\,[C_{\text{none}}(52) - C_{\text{scenario}}(52)]$. The background $b$ is assumed to be unaffected, so the outcome counts only cases of the season's outbreak. A negative value means the intervention *added* cases.
- **Uncertainty.** Every scenario is run for the same 2,000 posterior samples used in the predictive check (Section 3.4, generator `[SEED, 35]`); no new random numbers are drawn. Cases prevented are computed sample by sample against the no-intervention run of the same sample, so comparisons between scenarios are paired. Results are reported as the posterior median and 95% credible interval (2.5th–97.5th percentiles).
- **Reference week.** To express timing relative to the seasonal rise, the *rise week* is defined from the data: after the cases carried over from the 2025 season have declined to their lowest weekly count in the first half of 2026, it is the first week in which reported cases exceed twice the fitted background ($2b$).
- **Checks.** Runs extend to 104 weeks to confirm that prevented cases are not simply pushed into 2027, and a few scenarios are repeated with a 4× smaller RK4 step.

*Provenance: written with an AI coding agent.*

In [ ]:
# 5.1 Cases prevented for a grid of start weeks and reductions (sustained and 8-week campaign)
START_WEEKS = np.arange(1, 41)
REDUCTIONS = np.array([0.10, 0.25, 0.40])
CAMPAIGN_WEEKS = 8
HORIZON = 104                                    # simulate two years; 2026 = weeks 1-52

bg_median = np.median(sample["bg"])
low_week = int(weeks[np.argmin(y_obs[:26])])                       # end of the decline of the 2025 season
rise_week = int(weeks[low_week + np.argmax(y_obs[low_week:] > 2 * bg_median)])
print(f"early-year low: week {low_week}; rise week: {rise_week} "
      f"(first later week with reported cases > 2b = {2 * bg_median:.0f})")


def reported(p, horizon=HORIZON, steps_per_week=7):
    # expected reported outbreak cases (without background) in 2026 and over the full horizon
    w = simulate(p, horizon, steps_per_week)["weekly_reported"]
    return w[:52].sum(axis=0), w.sum(axis=0)


none_2026, none_all = reported(params_from(sample))


def cases_prevented(reductions, duration=None, horizon=HORIZON):
    """Reported cases prevented, shape (reduction, start week, posterior sample)."""
    shape = (len(reductions), START_WEEKS.size, none_2026.size)
    prev_2026, prev_all = np.empty(shape), np.empty(shape)
    for i, r in enumerate(reductions):
        for j in range(0, START_WEEKS.size, 10):                  # 10 start weeks x 2,000 samples per call
            start = START_WEEKS[j:j + 10, None] - 1.0              # week w starts at t = w - 1
            end = np.inf if duration is None else start + duration
            c_2026, c_all = reported(params_from(sample, start=start, end=end, reduction=r), horizon)
            prev_2026[i, j:j + 10] = none_2026 - c_2026
            prev_all[i, j:j + 10] = none_all - c_all
    return prev_2026, prev_all


t0 = time.perf_counter()
sustained, sustained_all = cases_prevented(REDUCTIONS)
campaign, campaign_all = cases_prevented(REDUCTIONS, duration=CAMPAIGN_WEEKS)
HEAT_REDUCTIONS = np.round(np.arange(0.05, 0.501, 0.05), 2)
heat, _ = cases_prevented(HEAT_REDUCTIONS, horizon=52)
print(f"{(2 * REDUCTIONS.size + HEAT_REDUCTIONS.size) * START_WEEKS.size} scenarios x {none_2026.size:,} "
      f"posterior samples in {time.perf_counter() - t0:.0f} s")
print(f"no intervention: {np.median(none_2026):,.0f} reported outbreak cases in 2026 "
      f"(95% CrI {np.percentile(none_2026, 2.5):,.0f}-{np.percentile(none_2026, 97.5):,.0f}), plus the background")


def summary(x):
    return np.median(x, axis=-1), np.percentile(x, 2.5, axis=-1), np.percentile(x, 97.5, axis=-1)


rows = []
for name, res in [("sustained", sustained), (f"{CAMPAIGN_WEEKS}-week campaign", campaign)]:
    for i, r in enumerate(REDUCTIONS):
        med, lo, hi = summary(res[i])
        best = med.argmax()
        best_per_sample = START_WEEKS[res[i].argmax(axis=0)]
        row = {"intervention": name, "reduction": f"{r:.0%}"}
        for w in [1, 11, rise_week - 4, rise_week, rise_week + 4, rise_week + 8, rise_week + 12]:
            row[f"start wk {w}"] = f"{med[w - 1]:,.0f} ({lo[w - 1]:,.0f} to {hi[w - 1]:,.0f})"
        row["best start wk (median)"] = f"{START_WEEKS[best]}: {med[best]:,.0f}"
        row["best wk 95% CrI"] = (f"{np.percentile(best_per_sample, 2.5):.0f}-"
                                   f"{np.percentile(best_per_sample, 97.5):.0f}")
        rows.append(row)
timing_table = pd.DataFrame(rows).set_index(["intervention", "reduction"])
print("\nReported cases prevented in 2026, median (95% CrI)")
print(timing_table.T.to_string())

# Checks: prevented cases cannot exceed the outbreak, are not merely deferred to 2027,
# and do not depend on the RK4 step
assert (sustained <= none_2026).all() and (campaign <= none_2026).all()
deferred = max(np.abs(sustained_all - sustained).max(), np.abs(campaign_all - campaign).max())
print(f"\nlargest change in cases prevented when 2027 is included: {deferred:.1f} cases")
assert deferred < 0.05 * np.abs(sustained).max(), "cases are being pushed into the next year"
step_err = 0.0
for r, start, end in [(0.40, 0.0, 8.0), (0.25, rise_week - 1.0, np.inf), (0.10, 25.0, 33.0)]:
    p = params_from(sample, start=start, end=end, reduction=r)
    coarse = none_2026 - reported(p, 52)[0]
    fine = reported(params_from(sample), 52, 28)[0] - reported(p, 52, 28)[0]
    step_err = max(step_err, np.abs(coarse - fine).max())
print(f"largest change with a 4x smaller RK4 step: {step_err:.2e} cases")
assert step_err < 1.0
print("PASS: results bounded by the outbreak size, not deferred to 2027, and independent of the step size")

In [ ]:
# Figures: cases prevented against start week (lines with 95% bands) and heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, res, title in [(axes[0], sustained, "Sustained until the end of 2026"),
                       (axes[1], campaign, f"{CAMPAIGN_WEEKS}-week campaign")]:
    for i, r in enumerate(REDUCTIONS):
        med, lo, hi = summary(res[i])
        ax.fill_between(START_WEEKS, lo, hi, color=f"C{i}", alpha=0.2)
        ax.plot(START_WEEKS, med, color=f"C{i}", lw=2, label=f"{r:.0%} reduction")
    ax.axvline(rise_week, color="k", ls="--", lw=0.8)
    ax.axvline(weeks[y_obs.argmax()], color="C3", ls=":", lw=1)
    ax.axhline(0, color="0.3", lw=0.8)
    ax.set(title=title, xlabel="Start week of the intervention (2026)")
axes[0].set_ylabel("Reported cases prevented in 2026")
axes[0].text(rise_week + 0.3, axes[0].get_ylim()[1] * 0.93, "rise week", fontsize=8)
axes[0].text(weeks[y_obs.argmax()] + 0.3, axes[0].get_ylim()[1] * 0.86, "observed peak", fontsize=8, color="C3")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Cases prevented by start week: posterior median and 95% credible band (2,000 samples)")
plt.tight_layout()
fig.savefig(FIG_DIR / "intervention_timing.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.2))
heat_med = np.median(heat, axis=-1)
im = ax.pcolormesh(START_WEEKS, HEAT_REDUCTIONS * 100, heat_med, cmap="viridis", shading="nearest")
cs = ax.contour(START_WEEKS, HEAT_REDUCTIONS * 100, heat_med, levels=[500, 1000, 2000, 4000, 8000],
                colors="w", linewidths=0.8)
ax.clabel(cs, fmt="%d", fontsize=8)
ax.axvline(rise_week, color="w", ls="--", lw=0.8)
ax.grid(False)
fig.colorbar(im, ax=ax, label="Median reported cases prevented in 2026")
ax.set(xlabel="Start week of a sustained intervention (2026)", ylabel="Transmission reduction r (%)",
       title="Median cases prevented by start week and strength (sustained intervention)")
fig.savefig(FIG_DIR / "intervention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 The cost of each week of delay

For a decision-maker the useful number is how many cases are lost for each week the start is postponed. For every posterior sample, the cases prevented when starting in week $w$ are compared with starting 4 weeks later, and the difference is divided by 4. Because both runs use the same sample, this paired difference is much more precise than the difference of the two intervals in the table above. A positive value means that delay costs cases; a negative value means that waiting would have prevented more.

In [ ]:
# 5.2 Extra reported cases per week of delay, over 4-week windows (paired, per posterior sample)
windows = [(1, 5), (rise_week - 8, rise_week - 4), (rise_week - 4, rise_week), (rise_week, rise_week + 4),
           (rise_week + 4, rise_week + 8), (rise_week + 8, rise_week + 12)]
rows = []
for name, res in [("sustained", sustained), (f"{CAMPAIGN_WEEKS}-week campaign", campaign)]:
    for i, r in enumerate(REDUCTIONS):
        row = {"intervention": name, "reduction": f"{r:.0%}"}
        for a, b in windows:
            cost = (res[i, a - 1] - res[i, b - 1]) / (b - a)
            med, lo, hi = summary(cost)
            row[f"wk {a} -> {b}"] = f"{med:,.0f} ({lo:,.0f} to {hi:,.0f})"
        rows.append(row)
delay_table = pd.DataFrame(rows).set_index(["intervention", "reduction"])
print("Extra reported cases per week of delay, median (95% CrI); negative = waiting prevents more")
print(delay_table.T.to_string())

**What the scenarios show** (seeded results; all numbers are reported cases in 2026, median and 95% CrI). Without an intervention the model gives 14,502 (13,157–15,985) outbreak cases in 2026 on top of the background. The rise week by the rule above is **week 21**; the observed peak was week 27.

1. **A strong sustained cut: earlier is clearly better.** With a 40% reduction kept up for the rest of the year, starting in week 5 prevents 7,554 (4,198–11,009) cases, starting in the rise week 4,689 (3,418–6,108), four weeks later 3,673 (3,040–4,321), and in week 29 only 1,324 (1,056–1,622). Each week of delay costs about **256 (86–460) cases per week between weeks 21 and 25, and 586 (485–684) per week between weeks 25 and 29**. Even before the rise, delaying from week 13 to week 17 costs 215 (75–362) cases per week. The best start, sample by sample, lies in weeks 2–7.
2. **Weak sustained cuts: starting early buys little.** With 10% or 25% the gain from starting early is small or negative: a 10% cut from week 1 prevents 165 (68–345) cases against 547 (405–737) from week 21. The best start is week 23 (95% interval 22–24 for 10%), just after the rise. Delay is costly only once the outbreak is under way: from week 25 to 29 each week costs 65 (49–83) cases at 10% and 246 (192–308) at 25%.
3. **Short campaigns must be timed to the outbreak.** An 8-week campaign started before about week 20 *adds* cases, e.g. −531 (−661 to −409) for a 40% campaign in week 1. The best start is week 26–27, just before the observed peak, preventing 2,138 cases at 40% (1,268 at 25%, 455 at 10%). Starting in week 21 prevents almost nothing at 40% (−4, −163 to 333); waiting from week 21 to 25 gains 419 (332–505) cases per week of waiting. Once the peak has passed, delay costs again: for a 40% campaign about 98 (−6 to 224) cases per week between weeks 25 and 29, and 269 (221–320) per week between weeks 29 and 33.

**Why timing matters this way.** Section 3.4 found that the 2026 outbreak ended because the small at-risk population ran out, while seasonal transmission was still rising towards its maximum in week 33. An intervention that is too weak or too short to stop the outbreak does not prevent its infections; it postpones them. A postponed outbreak runs into the stronger transmission later in the season and *overshoots* further, infecting more of the at-risk pool than it would have otherwise. An intervention near the peak does the opposite: it trims the overshoot, which is where the prevented cases come from. Only a cut strong enough to keep transmission low for the whole season (here about 40% or more) gains steadily from an early start. Runs extended into 2027 change the prevented cases by at most 158, so the prevented cases are not simply moved into the next year.

**Plain-English answer.** For the 2026 Colombo season, a sustained intervention that cut transmission by 40% would have prevented roughly a third of the outbreak's reported cases if started at the rise in week 21, and about half if started in the early months; every week of delay after the rise would have cost about 250–600 reported cases. Weaker or short measures are worth much less, and their benefit depends on hitting the outbreak itself (weeks 23–27) rather than on starting as early as possible.

These conclusions rest on the depletion mechanism, the single-cosine season and the homogeneous-mixing structure of the model; Section 5.3 tests how sensitive they are.

### 5.3 Sensitivity analysis

**Which inputs drive the answer?** Three outputs from Section 5.1–5.2 are followed:
- **Y1**: reported cases prevented by a sustained 40% cut started in the rise week;
- **Y2**: extra cases per week of delay between the rise week and four weeks later, for the same intervention;
- **Y3**: cases prevented by a 40%, 8-week campaign started in week 1 (negative in the main analysis). This output tests whether the finding that early short campaigns add cases is robust.

The inputs fall into three groups, and each needs a different method:

1. **Fixed inputs taken from the literature: incubation period $1/\sigma$ and infectious period $1/\gamma$.** Changing them alone would break the fit: the model would no longer reproduce the observed outbreak. So for each alternative value the model is **re-calibrated** and the outputs are recomputed over the new posterior. The ranges are 4 and 8 days for $1/\sigma$ (base 5.9 days [3]) and 4 and 7 days for $1/\gamma$ (base 5 days [4]).
   - These ranges are deliberately wide. The reproduction number inferred from an observed growth rate, and with it the predicted effect of control, depends on the assumed latent and infectious periods [6].
   - A **structural variant** is also run, with $1/\sigma$ = 15 days. It adds the 8–12 day extrinsic incubation in the mosquito [4] to the human incubation, and so approximates the longer generation time of a host–vector model inside the host-only structure.
   - The five re-calibrations run together: 8 chains each, warm-started from the main posterior, with 3,000 burn-in and 6,000 kept iterations thinned by 10 (generator `[SEED, 53]`). Convergence is checked by split-$\hat R$ within each group.
2. **Calibrated inputs ($\rho$, $a$, $s_0$, $R_e$, $\phi$, $I_0$).** Their plausible range is their posterior. The *one-at-a-time* view compares the outputs among posterior samples in the lowest and highest quarter of each parameter. The *global* measure is the partial rank correlation coefficient (PRCC) [7] between each parameter and the output across all 2,000 samples, with 95% bootstrap intervals (1,000 resamples, generator `[SEED, 54]`). The PRCC measures the monotone effect of one input after removing the effect of the others. With strongly correlated posterior parameters (such as $\rho$ and $s_0$) it should be read as a statement about the posterior, not about independent changes.
3. **The scenario input: reduction $r$**, set to 30% and 50% instead of 40%.

Every estimate is reported as the posterior median with a 95% credible interval (bootstrap interval for the PRCC).

In [ ]:
# 5.3 Sensitivity analysis: re-calibration for other incubation/infectious periods
def timing_outputs(theta, sigma=SIGMA_DEFAULT, gamma=GAMMA_DEFAULT, r=0.40):
    """Y1, Y2, Y3 (see text) for a batch of posterior samples theta, with the given periods (1/week)."""
    p = Params(N=N_COLOMBO, beta0=theta["Re"] * gamma / theta["s0"], amp=theta["amp"], phase=theta["phase"],
               rho=theta["rho"], s0=theta["s0"], i0=theta["i0"], e0=theta["i0"] * gamma / sigma,
               sigma=sigma, gamma=gamma)
    none = simulate(p, 52)["weekly_reported"].sum(axis=0)
    prevented = lambda **c: none - simulate(p.with_(**c), 52)["weekly_reported"].sum(axis=0)
    y1 = prevented(start=rise_week - 1.0, end=np.inf, reduction=r)
    y_later = prevented(start=rise_week + 3.0, end=np.inf, reduction=r)
    y3 = prevented(start=0.0, end=float(CAMPAIGN_WEEKS), reduction=r)
    return {"Y1": y1, "Y2": (y1 - y_later) / 4, "Y3": y3}


base_out = timing_outputs(sample)
OUTPUTS = {"Y1": "cases prevented, 40% from rise week", "Y2": "cases lost per week of delay",
           "Y3": "early 8-week campaign (week 1)"}

PERIODS = [("incubation 1/sigma = 4 d", 7 / 4, GAMMA_DEFAULT), ("incubation 1/sigma = 8 d", 7 / 8, GAMMA_DEFAULT),
           ("infectious 1/gamma = 4 d", SIGMA_DEFAULT, 7 / 4), ("infectious 1/gamma = 7 d", SIGMA_DEFAULT, 7 / 7),
           ("latent incl. mosquito, 1/sigma = 15 d", 7 / 15, GAMMA_DEFAULT)]
CHAINS = 8
sens_rng = np.random.default_rng([SEED, 53])
sig = np.repeat([s for _, s, _ in PERIODS], CHAINS)          # one (sigma, gamma) pair per chain
gam = np.repeat([g for _, _, g in PERIODS], CHAINS)
refit_post = seir_model(y_obs, SEIR_PRIORS, dict(FIXED, sigma=sig, gamma=gam), simulate, Params, MCMC_STEPS_PER_WEEK)
warm = sens_rng.choice(draws["Re"].size, sig.size, replace=False)
t0 = time.perf_counter()
refit = metropolis(refit_post, refit_post.to_z({n: draws[n][warm] for n in fit.names}),
                   n_burn=3_000, n_keep=6_000, rng=sens_rng, thin=10, reset_outliers=False)
print(f"{len(PERIODS)} re-calibrations x {CHAINS} chains x 9,000 iterations in {time.perf_counter() - t0:.0f} s")

period_out, refit_rows = {}, []
for g, (label, s_, g_) in enumerate(PERIODS):
    cols = slice(g * CHAINS, (g + 1) * CHAINS)
    rhat = max(split_rhat(refit.theta[n][:, cols]) for n in fit.names)
    assert rhat < 1.1, f"{label}: chains have not converged (split R-hat {rhat:.3f})"
    th = {n: refit.theta[n][:, cols].reshape(-1) for n in fit.names}
    k = sens_rng.choice(th["Re"].size, 1000, replace=False)
    th = {n: v[k] for n, v in th.items()}
    period_out[label] = timing_outputs(th, s_, g_)
    refit_rows.append({"assumption": label, "max split R-hat": f"{rhat:.3f}",
                       "Re": f"{np.median(th['Re']):.2f}", "peak Re": f"{np.median(th['Re'] * (1 + th['amp'])):.2f}",
                       "rho*s0": f"{np.median(th['rho'] * th['s0']):.4f}",
                       "median log-post": f"{np.median(refit.log_post[:, cols]):.1f}"})
refit_rows.insert(0, {"assumption": "base (5.9 d, 5 d)", "max split R-hat": f"{fit_table.rhat.max():.3f}",
                      "Re": f"{np.median(draws['Re']):.2f}",
                      "peak Re": f"{np.median(draws['Re'] * (1 + draws['amp'])):.2f}",
                      "rho*s0": f"{np.median(draws['rho'] * draws['s0']):.4f}",
                      "median log-post": f"{np.median(fit.log_post):.1f}"})
print(pd.DataFrame(refit_rows).set_index("assumption").to_string())
print("PASS: every re-calibration has converged (split R-hat < 1.1)")

In [ ]:
# One-at-a-time sensitivity table (all inputs) and global PRCC for the calibrated parameters
def fmt(x):
    return f"{np.median(x):,.0f} ({np.percentile(x, 2.5):,.0f} to {np.percentile(x, 97.5):,.0f})"


oat = {"base": base_out, **period_out}
for r in (0.30, 0.50):
    oat[f"reduction r = {r:.0%}"] = timing_outputs(sample, r=r)
for name in ["rho", "amp", "s0", "Re", "phase", "i0"]:
    q25, q75 = np.percentile(sample[name], [25, 75])
    for tag, mask in [(f"<= {q25:.3g}", sample[name] <= q25), (f">= {q75:.3g}", sample[name] >= q75)]:
        oat[f"{name} {tag} (posterior quarter)"] = {y: v[mask] for y, v in base_out.items()}
oat_table = pd.DataFrame({label: {OUTPUTS[y]: fmt(v[y]) for y in OUTPUTS} for label, v in oat.items()}).T
print("Outputs under each input setting: median (95% CrI)")
print(oat_table.to_string())


def prcc(X, y):
    """Partial rank correlation of each column of X with y."""
    R = np.argsort(np.argsort(np.column_stack([X, y]), axis=0), axis=0).astype(float)
    out = []
    for j in range(X.shape[1]):
        others = np.column_stack([np.ones(len(R)), np.delete(R[:, :-1], j, axis=1)])
        res_x = R[:, j] - others @ np.linalg.lstsq(others, R[:, j], rcond=None)[0]
        res_y = R[:, -1] - others @ np.linalg.lstsq(others, R[:, -1], rcond=None)[0]
        out.append(np.corrcoef(res_x, res_y)[0, 1])
    return np.array(out)


# rho and s0 enter as rho*s0 (well identified) and s0 (the direction along the rho-s0 ridge)
GLOBAL_INPUTS = {"Re": sample["Re"], "amp": sample["amp"], "phase": sample["phase"],
                 "rho*s0": sample["rho"] * sample["s0"], "s0": sample["s0"], "i0": sample["i0"]}
X = np.column_stack(list(GLOBAL_INPUTS.values()))
boot_rng = np.random.default_rng([SEED, 54])
boots = boot_rng.integers(0, len(X), size=(1000, len(X)))
prcc_rows = {}
for y in ["Y1", "Y2"]:
    est = prcc(X, base_out[y])
    bs = np.array([prcc(X[b], base_out[y][b]) for b in boots])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    prcc_rows[OUTPUTS[y]] = [f"{e:+.2f} ({l:+.2f} to {h:+.2f})" for e, l, h in zip(est, lo, hi)]
print("\nPRCC with 95% bootstrap interval (2,000 posterior samples, 1,000 resamples)")
print(pd.DataFrame(prcc_rows, index=list(GLOBAL_INPUTS)).to_string())

In [ ]:
# Tornado plots: change in the median output from the base case, with 95% CrIs at each end
pairs = [("incubation period", "incubation 1/sigma = 4 d", "incubation 1/sigma = 8 d"),
         ("infectious period", "infectious 1/gamma = 4 d", "infectious 1/gamma = 7 d"),
         ("reduction r (30% / 50%)", "reduction r = 30%", "reduction r = 50%")]
for name in ["rho", "amp", "s0", "Re", "phase", "i0"]:
    low, high = [k for k in oat if k.startswith(name + " ")]
    pairs.append((f"{name} (posterior quarters)", low, high))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, y in zip(axes, ["Y1", "Y2"]):
    base_med = np.median(base_out[y])
    order = sorted(pairs, key=lambda p: abs(np.median(oat[p[2]][y]) - np.median(oat[p[1]][y])))
    for row, (label, low, high) in enumerate(order):
        for key, colour, txt, dy in [(low, "C0", "low", -0.2), (high, "C3", "high", 0.2)]:
            v = oat[key][y]
            med, lo, hi = np.median(v), np.percentile(v, 2.5), np.percentile(v, 97.5)
            ax.barh(row + dy, med - base_med, left=base_med, height=0.38, color=colour, alpha=0.6,
                    label=f"{txt} value of the input" if row == 0 else None)
            ax.errorbar(med, row + dy, xerr=[[med - lo], [hi - med]], color=colour, capsize=2, lw=0.8)
    struct = period_out["latent incl. mosquito, 1/sigma = 15 d"][y]
    ax.plot(np.median(struct), len(order), "kD", label="structural variant (re-calibrated)")
    ax.errorbar(np.median(struct), len(order), xerr=[[np.median(struct) - np.percentile(struct, 2.5)],
                [np.percentile(struct, 97.5) - np.median(struct)]], color="k", capsize=3, lw=1)
    ax.axvline(base_med, color="k", lw=1)
    ax.set_yticks(range(len(order) + 1), [p[0] for p in order] + ["latent incl. mosquito (15 d)"])
    ax.set(xlabel=OUTPUTS[y], title=f"{y}: base median {base_med:,.0f}")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Sensitivity of the timing results (sustained 40% cut from the rise week): medians and 95% CrIs")
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(FIG_DIR / "sensitivity_tornado.png", dpi=150, bbox_inches="tight")
plt.show()

**Which inputs matter most** (seeded results: median and 95% CrI; base values Y1 = 4,689, Y2 = 256, Y3 = −531).

1. **The qualitative conclusions hold in every setting.**
   - *Y3 is negative everywhere:* an 8-week campaign in week 1 adds cases in all 20 settings in the table (the base case included), including the structural variant (−83, −134 to −47).
   - *Delay costs cases (Y2 > 0) in every setting except two,* where the interval includes zero: a weaker 30% cut (63, −5 to 173) and the structural variant (21, −5 to 77).
2. **Intervention strength has the largest effect** on the numbers. Y1 is 2,574 (1,867–3,499) at 30% and 7,927 (6,329–9,193) at 50%. The cost of delay grows from 63 to 670 (441–829) cases per week.
3. **Next come the assumed incubation and infectious periods.** These are the fixed literature inputs, not the fitted parameters.
   - *Incubation period:* with 4 days instead of 5.9 the rise-week intervention prevents 6,768 (5,364–8,350) cases; with 8 days, 3,182 (2,256–4,268).
   - *Infectious period:* 4 days gives 5,563 (4,216–7,044) and 7 days gives 3,457 (2,577–4,591).
   - *Why:* the re-calibrated models fit the data about equally well (median log-posterior −248.7 to −250.3, against −249.1). But a longer generation time needs a higher reproduction number to produce the same observed growth: peak $R_e$ rises from 2.64 to 3.64 between the 4- and 8-day incubation periods [6]. A 40% cut then leaves more transmission above the threshold, and prevents less.
4. **Fitted parameters within their posterior matter less.** Comparing the lowest and highest posterior quarters moves Y1 by about ±15%:
   - $R_e$: 5,455 against 4,044;
   - $a$: 5,373 against 4,079;
   - $\phi$: 5,360 against 4,120.

   Stronger or later seasonal transmission makes the outbreak harder to control and reduces the gain from acting at the rise. In the PRCC this shows as $a$ −0.68 and $R_e$ −0.59 for Y1, and $a$ −0.83 and $\phi$ −0.58 for Y2.
5. **The reporting rate does not matter on its own.** The lowest and highest posterior quarters of $\rho$ give the same Y1 (4,690 against 4,687), and the same holds for $s_0$ (4,618 against 4,748). Only the product $\rho s_0$ matters, because it fixes how many reported cases the at-risk pool can produce; its PRCC is the largest, +0.88 (0.86–0.89) for Y1.
   - The PRCC of $s_0$ with $\rho s_0$ held fixed (−0.44) does not show up when $s_0$ changes on its own in the posterior. This is the caution of Section 5.3 in practice: with correlated posterior parameters the PRCC is a conditional statement.
6. **The structural variant is the largest single uncertainty.** If the mosquito stage is folded into a 15-day latent period, the re-calibrated model needs a peak $R_e$ of 5.8. A 40% cut from the rise week then prevents only 998 (623–1,613) cases, and the cost of delay almost vanishes. This model fits the data worse (median log-posterior −257.1 against −249.1), so the data favour the shorter generation time. Even so, the size of the effect, though not its direction, depends on how mosquito transmission is represented. This is the main reason a host–vector extension is worthwhile.

**Summary.** The direction of every timing conclusion is robust: early short campaigns add cases, and a strong sustained cut gains from starting earlier. The *size* of the benefit is uncertain, mainly through the intervention's real strength and the assumed generation time of dengue, not through the uncertainty in the fitted parameters. Across the settings tested, Y1 ranges from about 0.7 to 1.4 times its base value for the assumed periods, and from 0.55 to 1.7 times for reductions of 30–50%. The numbers in Sections 5.1–5.2 are therefore best read as central estimates.

## 6. Discussion and limitations

*To be completed with the results.* The main limitations already known are the host-only structure (no explicit mosquito population), a single sinusoidal seasonal pattern, homogeneous mixing across the whole district, and suspected (not laboratory-confirmed) case counts with a constant reporting rate.

**Web demo reference value:** *to be stated once the calibrated results are available.*

## 7. References

1. National Dengue Control Unit, Ministry of Health, Sri Lanka, "Weekly Dengue Update," 2026, weeks 1–45. [Online]. Available: https://www.dengue.health.gov.lk/weekly-report/
2. Department of Census and Statistics, Sri Lanka, "Census of Population and Housing 2024," final report, 2026. [Online]. Available: https://www.statistics.gov.lk/Blog/Blog_details
3. M. Chan and M. A. Johansson, "The incubation periods of dengue viruses," *PLoS ONE*, vol. 7, no. 11, p. e50972, 2012, doi: 10.1371/journal.pone.0050972.
4. World Health Organization, "Dengue and severe dengue," fact sheet. [Online]. Available: https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue
5. E. A. Undurraga, Y. A. Halasa and D. S. Shepard, "Use of expansion factors to estimate the burden of dengue in Southeast Asia: a systematic analysis," *PLoS Neglected Tropical Diseases*, vol. 7, no. 2, p. e2056, 2013, doi: 10.1371/journal.pntd.0002056.
6. H. J. Wearing, P. Rohani and M. J. Keeling, "Appropriate models for the management of infectious diseases," *PLoS Medicine*, vol. 2, no. 7, p. e174, 2005, doi: 10.1371/journal.pmed.0020174.
7. S. Marino, I. B. Hogue, C. J. Ray and D. E. Kirschner, "A methodology for performing global uncertainty and sensitivity analysis in systems biology," *Journal of Theoretical Biology*, vol. 254, no. 1, pp. 178–196, 2008, doi: 10.1016/j.jtbi.2008.04.011.

In [ ]:
elapsed = time.perf_counter() - NOTEBOOK_START
print(f"Notebook ran in {elapsed:.1f} s")
assert elapsed < 600, "Run all should finish within 10 minutes"